[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/crewai/crewai_routing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">使用 Arize Phoenix 追踪 CrewAI - 路由工作流</h1>

In [ ]:
%pip install -q arize-phoenix opentelemetry-sdk opentelemetry-exporter-otlp crewai crewai_tools openinference-instrumentation-crewai

## 设置密钥与依赖

注意：运行此 Colab 需要：

*   OpenAI API 密钥 (https://openai.com/)
*   Serper API 密钥 (https://serper.dev/)
*   Phoenix API 密钥 (https://app.phoenix.arize.com/)

In [ ]:
import os
from getpass import getpass

# 尚未设置 API 密钥时提示用户输入
openai_key = os.getenv("OPENAI_API_KEY", "OPENAI_API_KEY")
serper_key = os.getenv("SERPER_API_KEY", "SERPER_API_KEY")

if openai_key == "OPENAI_API_KEY":
    openai_key = getpass("请输入 OPENAI_API_KEY： ")

if serper_key == "SERPER_API_KEY":
    serper_key = getpass("请输入 SERPER_API_KEY： ")

# 使用提供的密钥设置环境变量
os.environ["OPENAI_API_KEY"] = openai_key
os.environ["SERPER_API_KEY"] = serper_key

In [ ]:
if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(project_name="crewai-agents")

# 为 CrewAI 接入插桩

In [ ]:
from openinference.instrumentation.crewai import CrewAIInstrumentor

CrewAIInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

## 定义工作智能体

In [ ]:
import nest_asyncio
from crewai import Agent, Crew, Process, Task
from crewai.flow import Flow, listen, router, start
from pydantic import BaseModel

research_analyst = Agent(
    role="资深研究分析师",
    goal="收集并总结所请求主题的数据。",
    backstory="技术市场趋势专家。",
    allow_delegation=False,
)

content_strategist = Agent(
    role="技术内容策略师",
    goal="根据提供的研究编写文章提纲。",
    backstory="将数据转化为叙事的内容创作者。",
    allow_delegation=False,
)

接下来有两种实现方式：使用路由智能体，或使用 Flows 中的 ```@router()``` 装饰器，后者允许根据方法输出定义条件路由逻辑。

### 方案 1：定义路由智能体的逻辑，对查询分类并运行对应智能体

In [ ]:
routerAgent = Agent(
    role="Router",
    goal="将每条查询分类为 'research' 或 'content outline'。",
    backstory="内容工作流的分流机器人。",
    verbose=False,
)


def route(user_input: str, router):
    router_task = Task(
        description=user_input, agent=router, expected_output="一个词：'research' 或 'content'"
    )
    router_classify = Crew(
        agents=[router], tasks=[router_task], process=Process.sequential, verbose=False
    )
    router_results = router_classify.kickoff()
    return router_results


def type_of_task(router_results):
    if isinstance(router_results, list):
        result = router_results[0]
        result_text = result.text if hasattr(result, "text") else str(result)
    else:
        result_text = (
            router_results.text if hasattr(router_results, "text") else str(router_results)
        )
    task_type = result_text.strip().lower()

    return task_type


def working_agent(task_type, user_input: str):
    if "research" in task_type:
        agent = research_analyst
        label = "研究分析师"
    else:
        agent = content_strategist
        label = "内容策略师"

    work_task = Task(description=user_input, agent=agent, expected_output="智能体回复")
    worker_crew = Crew(agents=[agent], tasks=[work_task], process=Process.sequential, verbose=True)
    work_results = worker_crew.kickoff()
    if isinstance(work_results, list):
        output = work_results[0].text if hasattr(work_results[0], "text") else str(work_results[0])
    else:
        output = work_results.text if hasattr(work_results, "text") else str(work_results)

    print(f"\n=== 已路由到 {label} ({task_type}) ===\n{output}\n")
    return output

#### 运行示例

In [ ]:
# ─── 4) 运行示例 ─────────────────────────────────────────────────────────
for query in [
    "Please research the latest AI safety papers.",
    "Outline an article on AI safety trends.",
]:
    router_output = route(query, routerAgent)
    task_output = type_of_task(router_output)
    working_agent(task_output, query)

### 方案 2：通过 ```@router()``` 装饰器定义路由逻辑

In [ ]:
nest_asyncio.apply()


# 定义 Flow 状态
class RoutingState(BaseModel):
    query: str = ""
    route: str = ""


# 定义结构化 Flow
class RoutingFlow(Flow[RoutingState]):
    def __init__(self, query: str):
        super().__init__(state=RoutingState(query=query))

    @start()
    def handle_query(self):
        print(f"📥 收到查询： {self.state.query}")

    @router(handle_query)
    def decide_route(self):
        if "research" in self.state.query.lower():
            self.state.route = "research"
            return "research"
        else:
            self.state.route = "outline"
            return "outline"

    @listen("research")
    def run_research(self):
        task = Task(
            description=self.state.query,
            expected_output="AI 安全研究发现摘要",
            agent=research_analyst,
        )
        crew = Crew(
            agents=[research_analyst], tasks=[task], process=Process.sequential, verbose=True
        )
        crew.kickoff()

    @listen("outline")
    def run_content_strategy(self):
        task = Task(
            description=self.state.query,
            expected_output="有关给定主题的文章提纲",
            agent=content_strategist,
        )
        crew = Crew(
            agents=[content_strategist], tasks=[task], process=Process.sequential, verbose=True
        )
        crew.kickoff()

#### 运行示例

In [ ]:
queries = [
    "Please research the latest AI safety papers.",
    "Outline an article on AI safety trends.",
]

for query in queries:
    flow = RoutingFlow(query=query)
    flow.kickoff()

### 在 Phoenix 项目中查看运行产生的追踪与跨度。